# 10. Cross-Split Generalization

This notebook compares model behavior across multiple split schemes. The current pipeline already references several split columns, including `split_ho_pathway`, `split_random`, and `split_epigenetic_ood`. The purpose of this notebook is to determine whether the ranking of models is stable across different kinds of distribution shift.


In [2]:
!pip -q install anndata scanpy scikit-learn scipy pandas numpy torch pyarrow

from google.colab import drive
drive.mount('/content/drive')

import os
import sys
import urllib.request
from pathlib import Path

HELPER_DIR = Path("/content/drive/MyDrive/ChemDFM")
if str(HELPER_DIR) not in sys.path:
    sys.path.append(str(HELPER_DIR))

from chemdfm_notebook_helpers import *

DATA_PATH = Path("/content/drive/MyDrive/ChemDFM/data/raw/sciplex_complete_middle_subset.h5ad")
DATA_PATH.parent.mkdir(parents=True, exist_ok=True)

if not DATA_PATH.exists():
    print("Downloading SciPlex dataset...")
    URL = "https://f003.backblazeb2.com/file/chemCPA-datasets/sciplex_complete_middle_subset.h5ad"
    urllib.request.urlretrieve(URL, DATA_PATH)
    print("Download complete.")

print("PROJECT_ROOT =", PROJECT_ROOT)
print("DATA_PATH =", DATA_PATH)
print("Using device:", DEVICE)

Mounted at /content/drive
PROJECT_ROOT = /content/drive/MyDrive/ChemDFM
DATA_PATH = /content/drive/MyDrive/ChemDFM/data/raw/sciplex_complete_middle_subset.h5ad
Using device: cpu


In [3]:
from pathlib import Path
import json, os, random, pickle, warnings
from dataclasses import dataclass, asdict

import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

PROJECT_ROOT = Path("/content/drive/MyDrive/ChemDFM")
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"
EXTERNAL_DIR = DATA_DIR / "external"
RUNS_DIR = PROJECT_ROOT / "runs"
RESULTS_DIR = PROJECT_ROOT / "results"
REPORTS_DIR = PROJECT_ROOT / "reports"

for p in [DATA_DIR, RAW_DIR, INTERIM_DIR, PROCESSED_DIR, EXTERNAL_DIR, RUNS_DIR, RESULTS_DIR, REPORTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

DEFAULT_H5AD = RAW_DIR / "sciplex_complete_middle_subset.h5ad"

def save_json(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)

def map_split(x: str) -> str:
    x = str(x).lower()
    if "train" in x:
        return "train"
    if "test" in x or "val" in x:
        return "test"
    if "ood" in x:
        return "ood"
    return "drop"

print("PROJECT_ROOT =", PROJECT_ROOT)
print("DEFAULT_H5AD =", DEFAULT_H5AD)


PROJECT_ROOT = /content/drive/MyDrive/ChemDFM
DEFAULT_H5AD = /content/drive/MyDrive/ChemDFM/data/raw/sciplex_complete_middle_subset.h5ad


In [4]:
(RESULTS_DIR / 'cross_split').mkdir(parents=True, exist_ok=True)


In [5]:
import anndata as ad

DATA_PATH = DEFAULT_H5AD
assert DATA_PATH.exists(), f"Missing data file: {DATA_PATH}"

adata = ad.read_h5ad(DATA_PATH)
print(adata)
print("obs columns:", list(adata.obs.columns))

if "dose_val" in adata.obs.columns and "dose" not in adata.obs.columns:
    adata.obs["dose"] = adata.obs["dose_val"]

X = adata.X
if hasattr(X, "toarray"):
    X = X.toarray()
X = np.asarray(X, dtype=np.float32)


AnnData object with n_obs × n_vars = 354640 × 2000
    obs: 'cell_type', 'dose', 'dose_character', 'dose_pattern', 'g1s_score', 'g2m_score', 'pathway', 'pathway_level_1', 'pathway_level_2', 'product_dose', 'product_name', 'proliferation_index', 'replicate', 'size_factor', 'target', 'vehicle', 'batch', 'n_counts', 'dose_val', 'condition', 'drug_dose_name', 'cov_drug_dose_name', 'cov_drug', 'control', 'split_ho_pathway', 'split_tyrosine_ood', 'split_epigenetic_ood', 'split_cellcycle_ood', 'SMILES', 'split_ood_finetuning', 'split_ho_epigenetic', 'split_ho_epigenetic_all', 'split_random'
    var: 'id', 'num_cells_expressed-0-0', 'num_cells_expressed-1-0', 'num_cells_expressed-1', 'gene_id', 'in_lincs', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'all_DEGs', 'hvg', 'lincs_DEGs', 'neighbors', 'pca', 'umap'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    obsp: 'connectivities', 'distances'
obs columns: ['cell_type', 'dose', 'dose_character', 'dose_pattern', 'g1s_sco

In [6]:
candidate_split_cols = [c for c in adata.obs.columns if c.startswith("split")]
cross_split_plan = pd.DataFrame({"split_col": candidate_split_cols})
cross_split_plan.to_csv(RESULTS_DIR / "cross_split" / "split_candidates.csv", index=False)
cross_split_plan


,split_col
0,split_ho_pathway
1,split_tyrosine_ood
2,split_epigenetic_ood
3,split_cellcycle_ood
4,split_ood_finetuning
5,split_ho_epigenetic
6,split_ho_epigenetic_all
7,split_random


In [7]:
# Two usage modes are supported:
# 1) aggregate previously completed runs trained under different split columns,
# 2) programmatically re-run training loops with split_col swapped.
#
# The scaffold below supports run aggregation once such runs are available.
from pathlib import Path
CROSS_DIR = RESULTS_DIR / "cross_split"
CROSS_DIR.mkdir(parents=True, exist_ok=True)

records = []
for run_dir in RUNS_DIR.glob("*"):
    config_path = run_dir / "config_resolved.json"
    metrics_path = run_dir / "final_overall_metrics.csv"
    if config_path.exists() and metrics_path.exists():
        cfg = json.load(open(config_path))
        metrics = pd.read_csv(metrics_path)
        for _, row in metrics.iterrows():
            records.append({
                "run_name": run_dir.name,
                "split_col": cfg.get("split_col", "unknown"),
                "split": row["split"],
                "model": row.get("model", run_dir.name),
                "r2_top50": row.get("r2_top50", np.nan),
                "r2_full": row.get("r2_full", np.nan),
            })

cross_split_summary = pd.DataFrame(records)
cross_split_summary.to_csv(CROSS_DIR / "cross_split_summary.csv", index=False)
cross_split_summary.head()


,run_name,split_col,split,model,r2_top50,r2_full
0,residual_base,split_ho_pathway,test,ResidualDoseResponseModel,0.577561,0.636585
1,residual_base,split_ho_pathway,ood,ResidualDoseResponseModel,0.557015,0.614290
2,residual_cellaware_mmd,split_ho_pathway,test,ResidualDoseResponseModel_CellAwareMMD,0.575434,0.635105
3,residual_cellaware_mmd,split_ho_pathway,ood,ResidualDoseResponseModel_CellAwareMMD,0.558485,0.615382


A complete cross-split experiment should instantiate the same model family under each split definition and then compare the stability of performance rankings. This notebook provides the aggregation layer; training under additional split columns can be done by copying the training notebook and changing `split_col`, or by converting training into a shared script later.
